# Chapter 2: First Query in 10 Minutes

This notebook contains all the code examples from Chapter 2.

## Setup

Make sure you have the required packages installed:
```bash
pip install duckdb polars pyarrow jupyter pandas
```

## Version Check

In [ ]:
import duckdb
print(f'DuckDB {duckdb.__version__}')
# Should see 1.5.0 or higher

## Generate Sample Data (if needed)

Run this cell if you don't have `orders.parquet` yet:

In [ ]:
import polars as pl
import datetime

# 10M orders across 2 years
orders = pl.DataFrame({
    'order_id': range(1, 10_000_001),
    'customer_id': (pl.arange(1, 10_000_001, eager=True) % 500_000) + 1,
    'order_date': pl.date_range(
        datetime.date(2022, 1, 1),
        datetime.date(2023, 12, 31),
        interval='1d',
        eager=True
    ).sample(10_000_000, with_replacement=True),
    'amount': (pl.arange(1, 10_000_001, eager=True) % 500 + 20.0),
    'status': pl.Series(['completed', 'pending', 'cancelled', 'refunded'])\
        .sample(10_000_000, with_replacement=True)
})

# Write as Parquet
orders.write_parquet('orders.parquet', compression='snappy')
print(f"[OK] Generated {len(orders):,} rows -> orders.parquet")

## Your First Query

Query 10M rows of Parquet directly - no import, no loading:

In [ ]:
import duckdb

# Query Parquet directly - no import, no loading
con = duckdb.connect()

result = con.execute("""
    SELECT
        DATE_TRUNC('month', order_date) as month,
        status,
        COUNT(*) as orders,
        SUM(amount) as revenue,
        AVG(amount) as avg_order
    FROM read_parquet('orders.parquet')
    WHERE order_date >= '2023-01-01'
    GROUP BY 1, 2
    ORDER BY 1, 2
""").fetchdf()

print(result)

## Check Compression Ratio

In [ ]:
import os

file_size = os.path.getsize('orders.parquet') / 1_000_000
print(f"Compressed: {file_size:.0f} MB")
# Uncompressed in memory would be ~1.2 GB

## Query Plan

See how DuckDB applies filters before loading data:

In [ ]:
plan = con.execute("""
    EXPLAIN
    SELECT DATE_TRUNC('month', order_date) as month,
           COUNT(*) as orders
    FROM read_parquet('orders.parquet')
    WHERE order_date >= '2023-01-01'
    GROUP BY 1
""").fetchall()

# fetchall() returns (label, plan_text) tuples; print the plan text
print(plan[0][1])

## Smoke Test Checklist

In [ ]:
import duckdb
import polars as pl
import pyarrow.parquet as pq

# [OK] DuckDB can query Parquet
assert duckdb.execute("SELECT COUNT(*) FROM read_parquet('orders.parquet')").fetchone()[0] == 10_000_000
print("[OK] DuckDB can query Parquet")

# [OK] Polars can read Parquet lazily
df = pl.scan_parquet('orders.parquet')
assert df.select(pl.len()).collect().item() == 10_000_000
print("[OK] Polars can read Parquet lazily")

# [OK] PyArrow can read metadata without loading data
metadata = pq.read_metadata('orders.parquet')
print(f"[OK] PyArrow: {metadata.num_rows:,} rows in {metadata.num_row_groups} row groups")

print("\nStack verified. You're ready.")

## Quick Win Queries

### Top 10 Customers by Total Spend

In [ ]:
top_customers = con.execute("""
    SELECT customer_id,
           SUM(amount) as total_spent,
           COUNT(*) as order_count
    FROM read_parquet('orders.parquet')
    GROUP BY customer_id
    ORDER BY total_spent DESC
    LIMIT 10
""").fetchdf()

print(top_customers)

### Daily Order Volume with 7-Day Moving Average

In [ ]:
daily_orders = con.execute("""
    SELECT order_date,
           COUNT(*) as orders,
           AVG(COUNT(*)) OVER (
               ORDER BY order_date
               ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
           ) as rolling_avg_7d
    FROM read_parquet('orders.parquet')
    GROUP BY order_date
    ORDER BY order_date
""").fetchdf()

print(daily_orders.head(10))

### Revenue by Status (Pie Chart Data)

In [ ]:
revenue_by_status = con.execute("""
    SELECT status,
           SUM(amount) as revenue,
           ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2) as pct
    FROM read_parquet('orders.parquet')
    GROUP BY status
    ORDER BY revenue DESC
""").fetchdf()

print(revenue_by_status)